# 03. 배터리 수명 예측 모델링 및 성능 평가

본 노트북은 **Batch 1**을 학습/검증 데이터셋으로, **Batch 2**를 최종 테스트 데이터셋으로 활용하여 최적의 머신러닝 모델을 개발하고, **Nature Energy (2019) 원논문 성능(9.1% MAPE)**과 비교 분석합니다.

- **Train (Batch 1 CV)**: Batch 1 내 5-Fold Cross-Validation 평균 성능
- **Valid (Batch 1 Hold-out)**: 동일 프로토콜 분리를 보장한 Hold-out 검증 성능 (데이터 누수 방지)
- **Test (Batch 2)**: Batch 2 최종 평가 성능 및 배치 간 일반화 저하 분석


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('../src')
from preprocess import load_processed_data
from features import extract_features, FEATURE_SETS
from train import run_training_pipeline

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['AppleGothic', 'NanumGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False



## 1. 모델 학습 및 평가 파이프라인 실행


In [ ]:
df_results, b2_errors = run_training_pipeline(output_dir='../results')
display(df_results)


## 2. Reporting Format (for Regression) 성능 결과표


In [ ]:
# 원논문 대표 모델: Level 3 ElasticNet
enet_row = df_results[
    (df_results['Feature_Set'] == 'Level 3 (Full)') & 
    (df_results['Model'].str.contains('ElasticNet'))
].iloc[0]

# 최적 비선형 트리 모델: Level 1 RandomForest
rf_row = df_results[
    (df_results['Feature_Set'] == 'Level 1 (Variance)') & 
    (df_results['Model'].str.contains('RandomForest'))
].iloc[0]

def make_reporting_table(row, title):
    table_data = [
        {'구분': 'Train (Batch 1 CV)', 'MAPE (%)': f"{row['Train (Batch 1 CV)']:.2f}%", '비고': 'Batch 1 5-Fold CV 평균 성능'},
        {'구분': 'Valid (Batch 1 Hold-out)', 'MAPE (%)': f"{row['Valid (Batch 1 Hold-out)']:.2f}%", '비고': '프로토콜 분리 Hold-out 검증'},
        {'구분': 'Test (Batch 2)', 'MAPE (%)': f"{row['Test (Batch 2)']:.2f}%", '비고': 'Batch 2 최종 평가 성능'},
        {'구분': 'Gap (Train-Valid)', 'MAPE (%)': f"{row['Gap (Train-Valid)']:+.2f}%", '비고': '(+) : 과적합 의심'},
        {'구분': 'Gap (Valid-Test)', 'MAPE (%)': f"{row['Gap (Valid-Test)']:+.2f}%", '비고': '(+) : 배치간 일반화 저하 의심'},
        {'구분': 'Gap (Target-Test)', 'MAPE (%)': f"{row['Gap (Target-Test)']:+.2f}%", '비고': 'Target : 원논문 9.1%'}
    ]
    df_rep = pd.DataFrame(table_data)
    print(f'=== {title} ===')
    return df_rep

display(make_reporting_table(enet_row, '원논문 정규화 선형 모델: Level 3 Full (ElasticNet)'))
display(make_reporting_table(rf_row, '최적 머신러닝 모델: Level 1 Variance (RandomForest)'))



## 3. 예측 성능 시각화 (Actual vs Predicted & Residuals)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ElasticNet
axes[0].scatter(b2_errors['cycle_life'], b2_errors['Pred_ElasticNet_Full'], color='crimson', alpha=0.7, edgecolors='k', label='Batch 2 Cells')
axes[0].plot([300, 1200], [300, 1200], 'k--', label='Ideal 1:1 Line')
axes[0].set_xlabel('Actual Cycle Life (Batch 2)')
axes[0].set_ylabel('Predicted Cycle Life')
axes[0].set_title(f"Full Model (ElasticNet) - Test MAPE: {enet_row['Test (Batch 2)']:.2f}%")
axes[0].legend()

# RandomForest
axes[1].scatter(b2_errors['cycle_life'], b2_errors['Pred_RandomForest'], color='navy', alpha=0.7, edgecolors='k', label='Batch 2 Cells')
axes[1].plot([300, 1200], [300, 1200], 'k--', label='Ideal 1:1 Line')
axes[1].set_xlabel('Actual Cycle Life (Batch 2)')
axes[1].set_ylabel('Predicted Cycle Life')
axes[1].set_title(f"Variance Model (RandomForest) - Test MAPE: {rf_row['Test (Batch 2)']:.2f}%")
axes[1].legend()

plt.tight_layout()
plt.show()



## 4. 오류 분석 (Error Analysis) - 오차가 가장 큰 셀들의 특성


In [ ]:
print('🚨 예측 오차가 가장 큰 상위 5개 셀:')
display(b2_errors.head(5))
